#  Asistente Experto en Existencialismo, Logoterapia y Humanismo — RAG + Groq + LangGraph

**Proyecto Final — IA Generativa**

Este notebook implementa un agente de inteligencia artificial experto en **Existencialismo, Logoterapia y Humanismo**, capaz de responder preguntas usando su propia base de conocimiento vectorial (RAG), manteniendo memoria de conversación entre turnos.

**Stack tecnológico:**
- LLM: **Groq (`openai/gpt-oss-120b`)**
- Embeddings: **HuggingFace `sentence-transformers/all-MiniLM-L6-v2`** (local, gratis, sin límites de cuota)
- Base de conocimiento vectorial: **ChromaDB**
- Framework de agente: **LangGraph** (con LangChain como base)
- Entorno: **Jupyter Notebook / Google Colab**

---

## Índice
1. [Instalación de dependencias](#1)
2. [Configuración de la API Key de Groq](#2)
3. [Carga y preprocesamiento de documentos PDF](#3)
4. [Creación de la base de conocimiento vectorial (ChromaDB + HuggingFace Embeddings)](#4)
5. [Prueba de la colección vectorial](#5)
6. [Diseño del System Prompt](#6)
7. [Agente en LangGraph (RAG + Groq)](#7)
8. [Memoria de conversación](#8)
9. [Celda de interacción (chat)](#9)
10. [5 preguntas de ejemplo documentadas](#10)


<a id="1"></a>
## 1. Instalación de dependencias

Instalamos las librerías necesarias: LangChain, LangGraph, integración de Groq, embeddings de HuggingFace, ChromaDB y utilidades para leer PDFs.

In [13]:
# Ejecuta esta celda una sola vez (o si te falta alguna librería)
%pip install -q langchain langchain-groq langchain-huggingface langchain-chroma langchain-community langchain-text-splitters langgraph python-dotenv pypdf sentence-transformers

In [2]:
import os
import glob
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

print(" Librerías importadas correctamente")

/tmp/ipykernel_931/4210986065.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


 Librerías importadas correctamente


<a id="2"></a>
## 2. Configuración de la API Key de Groq

**⚠️ Buenas prácticas:** la API key nunca se escribe directamente en el código. La cargamos desde un archivo `.env` (o desde una variable de entorno, por ejemplo `os.environ` en Colab con `google.colab.userdata`).

Crea un archivo `.env` en la misma carpeta que este notebook con el siguiente contenido:

```
GROQ_API_KEY=tu_api_key_aqui
```

Puedes obtener tu API key gratuita en [Groq Console](https://console.groq.com/keys).

Los embeddings ahora se generan **localmente con HuggingFace**, así que no necesitas ninguna API key para esa parte.

In [3]:
# Carga de API Key: compatible con entorno local (.env) y Google Colab
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Si no se encuentra en .env local, intentar desde los secretos de Google Colab
if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
        os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    except ImportError:
        pass

assert GROQ_API_KEY, "❌ No se encontró GROQ_API_KEY. Configúrala en tu archivo .env o en los Secretos de Colab."
print("✅ API key de Groq cargada correctamente")


 API key de Groq cargada correctamente


<a id="3"></a>
## 3. Carga y preprocesamiento de documentos PDF

**Dominio elegido: Existencialismo, Logoterapia y Humanismo** 🧭

Coloca tus archivos PDF (mínimo 3 documentos, o el equivalente a ~20 páginas) dentro de la carpeta `./documentos_existencialismo/`. Pueden ser, por ejemplo:
- Obras o resúmenes de filósofos existencialistas (Kierkegaard, Sartre, Camus, Heidegger, Jaspers, Simone de Beauvoir...)
- Textos sobre logoterapia y la búsqueda de sentido (Viktor Frankl, "El hombre en busca de sentido", "La voluntad de sentido"...)
- Artículos o apuntes sobre psicología humanista (Carl Rogers, Abraham Maslow, Rollo May, Erich Fromm...)
- Textos que aborden conceptos clave: libertad, angustia, absurdo, autenticidad, responsabilidad, sentido de la vida, autorrealización

El siguiente código carga todos los PDFs de la carpeta y los segmenta en fragmentos (*chunks*) manejables para el modelo de embeddings.


In [4]:
# Detectar carpeta de documentos (soporta './documentos_filosofia' en local y './documentos_existencialismo')
if os.path.exists("./documentos_filosofia") and glob.glob("./documentos_filosofia/*.pdf"):
    DOCS_DIR = "./documentos_filosofia"
elif os.path.exists("./documentos_existencialismo") and glob.glob("./documentos_existencialismo/*.pdf"):
    DOCS_DIR = "./documentos_existencialismo"
else:
    DOCS_DIR = "./documentos_filosofia"
    os.makedirs(DOCS_DIR, exist_ok=True)

rutas_pdf = glob.glob(os.path.join(DOCS_DIR, "*.pdf"))
assert len(rutas_pdf) > 0, f"❌ No se encontraron PDFs en {DOCS_DIR}. Coloca al menos 3 archivos .pdf ahí."

print(f"📄 Se encontraron {len(rutas_pdf)} PDFs en {DOCS_DIR}:")
for r in rutas_pdf:
    print("  -", os.path.basename(r))

# 1. Cargar todas las páginas de todos los PDFs
documentos = []
for ruta in rutas_pdf:
    loader = PyPDFLoader(ruta)
    documentos.extend(loader.load())

print(f"\n📚 Total de páginas cargadas: {len(documentos)}")

# 2. Segmentar en chunks manejables
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = splitter.split_documents(documentos)

print(f"✂️  Total de fragmentos (chunks) generados: {len(chunks)}")


 Se encontraron 3 PDFs:
  - El_existencialismo_es_un_humanismo_JeanPaul_Sartre-11-35.pdf
  - El_hombre_en_busca_de_sentido_Viktor_Frankl-96-128.pdf
  - El_mito_de_Sisifo_Albert_Camus-11-45.pdf

 Total de páginas cargadas: 93
  Total de fragmentos (chunks) generados: 267


<a id="4"></a>
## 4. Creación de la base de conocimiento vectorial (ChromaDB + HuggingFace Embeddings)

Usamos el modelo de embeddings local `sentence-transformers/all-MiniLM-L6-v2` de HuggingFace para convertir cada fragmento de texto en un vector, y los indexamos en una colección persistente de **ChromaDB**. Al ser local, no consume cuota de ninguna API y funciona sin conexión una vez descargado el modelo.

In [5]:
# 1. Configuración de Embeddings con HuggingFace (gratis y local)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Detectar base de datos persistente local o de existencialismo
if os.path.exists("./chroma_db_filosofia"):
    CHROMA_DIR = "./chroma_db_filosofia"
    COLLECTION_NAME = "filosofia_kb"
else:
    CHROMA_DIR = "./chroma_db_existencialismo"
    COLLECTION_NAME = "existencialismo_kb"

# 2. Validar si la carpeta existe y tiene archivos reales (evitando archivos ocultos del sistema)
archivos_reales = [f for f in os.listdir(CHROMA_DIR) if not f.startswith('.')] if os.path.exists(CHROMA_DIR) else []

# 3. Cargar o crear la base de datos vectorial
if archivos_reales:
    print(f"♻️ Cargando base de conocimiento vectorial existente desde {CHROMA_DIR}...")
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )
else:
    print(f"🆕 Creando nueva base de conocimiento vectorial e indexando documentos en {CHROMA_DIR}...")
    assert len(chunks) > 0, "No hay fragmentos para indexar. Verifica que colocaste PDFs en la carpeta."

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR,
    )

print(f"✅ Colección '{COLLECTION_NAME}' lista con {vectorstore._collection.count()} vectores")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Creando nueva base de conocimiento vectorial e indexando documentos...
 Colección 'existencialismo_kb' lista con 267 vectores


<a id="5"></a>
## 5. Prueba de la colección vectorial

Antes de conectar el agente, verificamos que la búsqueda semántica en ChromaDB devuelve resultados relevantes.

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

consulta_prueba = "¿Qué es la logoterapia según Viktor Frankl?"  # 🔧 cambia esta consulta según tus documentos
resultados = retriever.invoke(consulta_prueba)

print(f" Consulta: {consulta_prueba}\n")
for i, r in enumerate(resultados, 1):
    fuente = os.path.basename(r.metadata.get('source', '?'))
    pagina = r.metadata.get('page', '?')
    if isinstance(pagina, int):
        pagina = pagina + 1

    print(f"--- Resultado {i} (fuente: {fuente}, página: {pagina}) ---")
    print(r.page_content[:300].strip(), "...\n")

 Consulta: ¿Qué es la logoterapia según Viktor Frankl?

--- Resultado 1 (fuente: El_mito_de_Sisifo_Albert_Camus-11-45.pdf, página: 19) ---
punto de su esfuerzo el hombre se halla ante lo irracional. Siente en sí
mismo su deseo de dicha y de razón. Lo absurdo nace de esta confrontación
entre el llamamiento humano y el silencio irrazonable del mundo. Esto es lo
que no hay que olvidar. A esto es a lo que hay que aferrarse, puesto que
toda ...

--- Resultado 2 (fuente: El_mito_de_Sisifo_Albert_Camus-11-45.pdf, página: 8) ---
hombre. Los amantes lo saben muy bien. Pero si esa respuesta es sincera, si
traduce ese singular estado del alma en el cual el vacío se hace elocuente,
en el que la cadena de los gestos cotidianos se rompe, en el cual el corazón
busca en vano el eslabón que la reanuda, entonces es el primer signo de ...

--- Resultado 3 (fuente: El_mito_de_Sisifo_Albert_Camus-11-45.pdf, página: 8) ---
enumerarse las consecuencias y sólo el clima puede hacerse sentir.
Quizá podamos alca

<a id="6"></a>
## 6. Diseño del System Prompt

### System prompt elegido

```
Eres "Viktor", un asistente experto en Existencialismo, Logoterapia y Humanismo. Tu
función es ayudar a estudiantes, pacientes en procesos terapéuticos y curiosos a
comprender autores, conceptos y herramientas de estas tres corrientes, basándote
siempre en el CONTEXTO recuperado de la base de conocimiento documental.

Reglas de comportamiento:
1. Responde SIEMPRE en español, con un tono cálido, cercano y reflexivo, propio de
   quien acompaña a otra persona a pensar sobre su propia vida (como Viktor Frankl
   en sus escritos), sin dejar de ser riguroso conceptualmente.
2. Basa tus respuestas principalmente en el CONTEXTO proporcionado. Si el contexto no
   contiene información suficiente para responder con confianza, dilo explícitamente
   ("No tengo información suficiente en mi base documental sobre esto") en lugar de
   inventar datos o citas.
3. Cuando cites ideas de un autor o corriente (Kierkegaard, Sartre, Camus, Heidegger,
   Frankl, Rogers, Maslow, Fromm, etc.), intenta mencionar de qué fuente/documento
   provienen si es relevante.
4. Puedes complementar el contexto con tu conocimiento general de existencialismo,
   logoterapia y humanismo solo cuando sea necesario para dar coherencia a la
   explicación, pero deja claro cuándo estás haciendo esto ("más allá de mis
   documentos...").
5. Conecta los conceptos con la experiencia humana concreta (libertad, angustia,
   absurdo, autenticidad, responsabilidad, búsqueda de sentido, autorrealización)
   en lugar de quedarte solo en la teoría abstracta.
6. IMPORTANTE: no eres un terapeuta ni sustituyes una terapia profesional. Si el
   usuario plantea una situación de sufrimiento personal profundo, angustia vital
   intensa, o menciona ideas de autolesión o suicidio, respóndele con empatía,
   anímale a buscar apoyo profesional o líneas de ayuda, y no continúes solo con
   contenido teórico.
7. Mantén coherencia con el historial de la conversación: si el usuario hace
   referencia a algo dicho anteriormente, respóndele teniendo en cuenta ese
   contexto previo.
8. Evita respuestas excesivamente largas: sé conciso, estructura en párrafos claros
   o listas cuando ayude a la comprensión.
```

### Justificación de las decisiones de diseño

- **Rol y tono ("Viktor", en referencia a Viktor Frankl):** darle una identidad concreta y un tono cálido/reflexivo es coherente con el dominio: el existencialismo y la logoterapia tratan sobre la experiencia humana vivida, no solo sobre teoría abstracta.
- **Prioridad al CONTEXTO recuperado:** es el núcleo de un sistema RAG — el prompt obliga al modelo a fundamentar sus respuestas en los documentos y a admitir cuando no tiene información suficiente, reduciendo alucinaciones.
- **Conexión con la experiencia humana concreta:** encaja con la naturaleza misma de estas corrientes, que nacen de la reflexión sobre la vida vivida (Frankl escribió su obra fundamental desde su experiencia en campos de concentración).
- **Regla de derivación a apoyo profesional:** dado que el dominio toca temas sensibles (sentido de la vida, sufrimiento, angustia existencial), es imprescindible que el asistente no actúe como sustituto de terapia y sepa remitir a ayuda profesional cuando corresponda.
- **Coherencia con el historial:** necesario para que la memoria de conversación (sección 8) tenga sentido en la práctica.


In [7]:
SYSTEM_PROMPT = """Eres "Viktor", un asistente experto en Existencialismo, Logoterapia y Humanismo. Tu
función es ayudar a estudiantes, personas en procesos de autoconocimiento y curiosos a
comprender autores, conceptos y herramientas de estas tres corrientes, basándote
siempre en el CONTEXTO recuperado de la base de conocimiento documental.

Reglas de comportamiento:
1. Responde SIEMPRE en español, con un tono cálido, cercano y reflexivo, propio de
   quien acompaña a otra persona a pensar sobre su propia vida (como Viktor Frankl
   en sus escritos), sin dejar de ser riguroso conceptualmente.
2. Basa tus respuestas principalmente en el CONTEXTO proporcionado. Si el contexto no
   contiene información suficiente para responder con confianza, dilo explícitamente
   ("No tengo información suficiente en mi base documental sobre esto") en lugar de
   inventar datos o citas.
3. Cuando cites ideas de un autor o corriente (Kierkegaard, Sartre, Camus, Heidegger,
   Frankl, Rogers, Maslow, Fromm, etc.), intenta mencionar de qué fuente/documento
   provienen si es relevante.
4. Puedes complementar el contexto con tu conocimiento general de existencialismo,
   logoterapia y humanismo solo cuando sea necesario para dar coherencia a la
   explicación, pero deja claro cuándo estás haciendo esto ("más allá de mis
   documentos...").
5. Conecta los conceptos con la experiencia humana concreta (libertad, angustia,
   absurdo, autenticidad, responsabilidad, búsqueda de sentido, autorrealización)
   en lugar de quedarte solo en la teoría abstracta.
6. IMPORTANTE: no eres un terapeuta ni sustituyes una terapia profesional. Si el
   usuario plantea una situación de sufrimiento personal profundo, angustia vital
   intensa, o menciona ideas de autolesión o suicidio, respóndele con empatía,
   anímale a buscar apoyo profesional o líneas de ayuda, y no continúes solo con
   contenido teórico.
7. Mantén coherencia con el historial de la conversación: si el usuario hace
   referencia a algo dicho anteriormente, respóndele teniendo en cuenta ese
   contexto previo.
8. Evita respuestas excesivamente largas: sé conciso, estructura en párrafos claros
   o listas cuando ayude a la comprensión.
"""

print(SYSTEM_PROMPT)


Eres "Viktor", un asistente experto en Existencialismo, Logoterapia y Humanismo. Tu
función es ayudar a estudiantes, personas en procesos de autoconocimiento y curiosos a
comprender autores, conceptos y herramientas de estas tres corrientes, basándote
siempre en el CONTEXTO recuperado de la base de conocimiento documental.

Reglas de comportamiento:
1. Responde SIEMPRE en español, con un tono cálido, cercano y reflexivo, propio de
   quien acompaña a otra persona a pensar sobre su propia vida (como Viktor Frankl
   en sus escritos), sin dejar de ser riguroso conceptualmente.
2. Basa tus respuestas principalmente en el CONTEXTO proporcionado. Si el contexto no
   contiene información suficiente para responder con confianza, dilo explícitamente
   ("No tengo información suficiente en mi base documental sobre esto") en lugar de
   inventar datos o citas.
3. Cuando cites ideas de un autor o corriente (Kierkegaard, Sartre, Camus, Heidegger,
   Frankl, Rogers, Maslow, Fromm, etc.), intenta m

<a id="7"></a>
## 7. Agente en LangGraph (RAG + Groq)

Construimos un grafo con dos nodos:
1. **`retrieve`**: recupera los fragmentos más relevantes de ChromaDB para la última pregunta del usuario.
2. **`generate`**: construye el prompt (system prompt + contexto recuperado + historial de conversación) y genera la respuesta con Groq (Llama 3 70B).

El estado del grafo (`MessagesState`) mantiene la lista de mensajes de la conversación (humano/IA), lo cual es la base sobre la que añadiremos memoria persistente en el siguiente paso.

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",  # Modelo actualizado a openai/gpt-oss-120b
    api_key=GROQ_API_KEY,
    temperature=0.3,
)

class AgentState(MessagesState):
    """Estado del grafo: hereda 'messages' de MessagesState y añade el contexto recuperado."""
    context: str


def retrieve_node(state: AgentState):
    """Recupera de ChromaDB los fragmentos relevantes para la última pregunta del usuario."""
    last_question = state["messages"][-1].content
    docs = retriever.invoke(last_question)
    contexto = "\n\n".join(
        f"[Fuente: {d.metadata.get('source', '?')}] {d.page_content}" for d in docs
    )
    return {"context": contexto}


def generate_node(state: AgentState):
    """Genera la respuesta usando Groq, el system prompt, el contexto RAG y el historial."""
    contexto = state.get("context", "")

    prompt_con_contexto = (
        f"CONTEXTO recuperado de la base de conocimiento:\n{contexto}\n\n"
        f"Responde la última pregunta del usuario usando el CONTEXTO anterior y el "
        f"historial de la conversación cuando sea relevante."
    )

    mensajes = [SystemMessage(content=SYSTEM_PROMPT)]
    mensajes.extend(state["messages"][:-1])  # historial previo
    mensajes.append(SystemMessage(content=prompt_con_contexto))
    mensajes.append(state["messages"][-1])  # última pregunta del usuario

    respuesta = llm.invoke(mensajes)
    return {"messages": [respuesta]}


# Construcción del grafo
graph_builder = StateGraph(AgentState)
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("generate", generate_node)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

print("Grafo de LangGraph definido (retrieve -> generate)")

Grafo de LangGraph definido (retrieve -> generate)


<a id="8"></a>
## 8. Memoria de conversación

Usamos `MemorySaver`, el *checkpointer* de LangGraph, para persistir el estado (incluyendo todo el historial de mensajes) asociado a un `thread_id`. Así, cada vez que invocamos el grafo con el mismo `thread_id`, el agente recuerda automáticamente los turnos anteriores.

In [9]:
memory = MemorySaver()
agent = graph_builder.compile(checkpointer=memory)

print("Agente compilado con memoria de conversación (checkpointer=MemorySaver)")

Agente compilado con memoria de conversación (checkpointer=MemorySaver)


<a id="9"></a>
## 9. Celda de interacción (chat)

Ejecuta la siguiente celda para chatear con el agente experto en filosofía de forma interactiva. Escribe `salir` para terminar la conversación.

In [10]:
def preguntar(pregunta: str, thread_id: str = "conversacion-demo"):
    """Envía una pregunta al agente manteniendo memoria según el thread_id."""
    config = {"configurable": {"thread_id": thread_id}}
    resultado = agent.invoke({"messages": [HumanMessage(content=pregunta)]}, config=config)
    return resultado["messages"][-1].content


# --- Demostración de que la memoria funciona ---
# Pregunta 1: introducimos un tema
respuesta_1 = preguntar("¿Quién fue Viktor Frankl y qué es la logoterapia?")
print("Usuario: ¿Quién fue Viktor Frankl y qué es la logoterapia?")
print("Viktor:", respuesta_1)
print("\n" + "=" * 80 + "\n")

# Pregunta 2: hace referencia directa a la respuesta anterior (sin repetir el nombre)
respuesta_2 = preguntar("¿Y cómo desarrolló él esas ideas durante su cautiverio?")
print("Usuario: ¿Y cómo desarrolló él esas ideas durante su cautiverio?")
print("Viktor:", respuesta_2)
print("\nNótese que el agente entendió que 'él' se refiere a Viktor Frankl gracias a la memoria de conversación.")


Usuario: ¿Quién fue Viktor Frankl y qué es la logoterapia?
Viktor: **Viktor Frankl** (1905‑1997) fue un neurólogo y psiquiatra austriaco, sobreviviente del Holocausto, cuya experiencia en los campos de concentración influyó profundamente en su pensamiento. En sus escritos, sobre todo en *El hombre en busca de sentido* (1946), Frankl plantea que la motivación esencial del ser humano no es la búsqueda del placer (como proponía Freud) ni la voluntad de poder (como sostenía Adler), sino la **búsqueda de sentido**.

### ¿Qué es la logoterapia?

La logoterapia es la escuela psicológica que fundó Frankl y que se basa en tres ideas centrales:

| Pilar | Descripción |
|-------|-------------|
| **Voluntad de sentido** | Cada persona siente una necesidad innata de encontrar un propósito que dé coherencia a su vida, incluso en circunstancias extremas. |
| **Libertad de actitud** | Frente a cualquier situación —por muy dolorosa o limitada— siempre queda la libertad de decidir cómo responder, es dec

In [11]:
def chat_interactivo(thread_id: str = "chat-interactivo"):
    print("Chat con Viktor, tu asistente experto en Existencialismo, Logoterapia y Humanismo")
    print("   (escribe 'salir' para terminar)\n")
    while True:
        pregunta = input("Tú: ")
        if pregunta.strip().lower() in ("salir", "exit", "quit"):
            print("¡Hasta luego!")
            break
        respuesta = preguntar(pregunta, thread_id=thread_id)
        print("Viktor:", respuesta, "\n")

# Descomenta la siguiente línea para iniciar el chat interactivo
# chat_interactivo()


<a id="10"></a>
## 10. Preguntas de ejemplo documentadas

A continuación, 5 preguntas de ejemplo junto con las respuestas generadas por el agente, demostrando el uso de RAG y memoria.

> **Nota:** adapta estas preguntas a los documentos de existencialismo, logoterapia y humanismo que hayas cargado en `./documentos_existencialismo/`.


In [12]:
import time

def preguntar_con_reintentos(pregunta: str, thread_id: str, max_reintentos: int = 5, espera_inicial: float = 8.0):
    """Igual que preguntar(), pero con reintentos y backoff exponencial si Groq
    devuelve un error de rate limit (429 - tokens por minuto excedidos)."""
    for intento in range(max_reintentos):
        try:
            return preguntar(pregunta, thread_id=thread_id)
        except Exception as e:
            mensaje = str(e).lower()
            es_rate_limit = "429" in mensaje or "rate_limit" in mensaje or "rate limit" in mensaje
            if es_rate_limit and intento < max_reintentos - 1:
                espera = espera_inicial * (2 ** intento)
                print(f"⏳ Límite de tasa alcanzado (intento {intento + 1}/{max_reintentos}). "
                      f"Esperando {espera:.0f}s antes de reintentar...")
                time.sleep(espera)
            else:
                raise
    raise RuntimeError("No se pudo obtener respuesta tras varios reintentos.")


preguntas_ejemplo = [
    "¿Qué es el existencialismo?",
    "¿Qué significa el absurdo según Albert Camus?",
    "¿Por qué Viktor Frankl dice que el ser humano necesita encontrar un sentido en su vida?",
    "¿Quién fue Sísifo y por qué Camus utiliza su historia como símbolo?",
    "¿Qué diferencia hay entre buscar el placer y buscar un propósito en la vida según Frankl?",
]

thread_demo = "demo-5-preguntas"

for i, pregunta in enumerate(preguntas_ejemplo, 1):
    respuesta = preguntar_con_reintentos(pregunta, thread_id=thread_demo)
    print(f"### Pregunta {i}")
    print("Usuario:", pregunta)
    print("Viktor:", respuesta)
    print("\n" + "-" * 80 + "\n")

    # Pequeña pausa entre preguntas para no saturar el límite de tokens/minuto de Groq
    if i < len(preguntas_ejemplo):
        time.sleep(5)


### Pregunta 1
Usuario: ¿Qué es el existencialismo?
Viktor: **El existencialismo** es una doctrina filosófica que coloca al ser humano, con su **libertad, responsabilidad y subjetividad**, en el centro de la reflexión sobre la vida.  

Según Jean‑Paul Sartre, en *El existencialismo es un humanismo*:

* “Entendemos por existencialismo una doctrina que **hace posible la vida humana** y que, por otra parte, declara que **toda verdad y toda acción implican un medio y una subjetividad humana**”【./documentos_existencialismo/El_existencialismo_es_un_humanismo_JeanPaul_Sartre-11-35.pdf】.  

* El existencialismo no se reduce a un ateísmo que busca demostrar que Dios no existe; más bien afirma que **“aunque Dios existiera, esto no cambiaría”** la necesidad de que el hombre se encuentre a sí mismo y reconozca que **nada puede salvarlo de sí mismo**, ni siquiera una prueba de la existencia divina【./documentos_existencialismo/El_existencialismo_es_un_humanismo_JeanPaul_Sartre-11-35.pdf】.  

* En au

---
## Resumen del proyecto

- Base de conocimiento vectorial creada con **ChromaDB + HuggingFace Embeddings** a partir de PDFs de Existencialismo, Logoterapia y Humanismo.
- Agente **LangGraph** que combina recuperación (RAG) y generación con **Groq (`openai/gpt-oss-120b`)**.
- **System prompt** personalizado y justificado ("Viktor", experto en existencialismo, logoterapia y humanismo).
- **Memoria de conversación** mediante `MemorySaver` (checkpointer de LangGraph).
- Celda de **chat interactivo** + 5 preguntas de ejemplo documentadas.

**Próximos pasos sugeridos:** agregar más documentos (obras de Frankl, Sartre, Camus, Rogers, Maslow...) para ampliar la base de conocimiento, evaluar la calidad de las respuestas con más preguntas de prueba, y ajustar `chunk_size`/`k` del retriever según los resultados obtenidos.
